# General Settings

In [108]:
import os
import pandas as pd

pd.set_option('display.max_columns', None)

# Import and Load
Load the raw data from the *Parametric Analysis* Sheet of an Excel file into a Pandas Data Frame using the Openpyxl engine. Überspringe die ersten beiden Zeilen des Excel Sheets, da diese sowieso keine relevanten Information enthalten

In [109]:
raw_data_path = os.path.join(
    os.path.dirname(os.getcwd()), "data", "raw", "MCDI_Overview_10-04-2025.xlsx"
)

df = pd.read_excel(
    raw_data_path, sheet_name="Parametric Analysis", engine="calamine", skiprows=2
)

df.head()

,Unnamed: 0,Test #,Mol.\r\n(g/L),H2O\r\n,pH\r\n,Time\r\n(s),Flow\r\n (L/min),Volt. Lim.\r\n(V),Curr. Lim.\r\n(A),Check,EC In \r\n(mS/cm),EC In Dev. (%),EC Pure\r\n(mS/cm),Remov. \r\n(%),SEC \r\n(kWh/m³),Spalte1,Ry \r\n(%),Min EC Pure (mS/cm),Max. Remov. (%),Avg. EC In (mS/cm),Avg. EC Pure (mS/cm),pH Pure,pH Conc.,Test Remarks,Unnamed: 24,Unnamed: 25
0,0 - General Tests,0.1,1.0,DI,-,95/15/80,"0,45/0,5/0,5","1,95/1,5/1,5",60/50/55,Valid,4.42,0.014712,1.28,71.0,-,NaN,40.0,0.082,98.1,4.06,0.09,-,-,Met all the criterias.,NaN,NaN
1,NaN,0.2,1.0,DI,-,95/15/80,"0,35/0,4/0,4","1,95/1,5/1,5",60/50/55,Valid,4.21,0.061525,1.82,57.0,-,NaN,41.0,0.134,96.8,4.15,0.16,-,-,NaN,NaN,NaN
2,NaN,0.3,1.0,DI,-,95/15/80,"0,3/0,3/0,3","1,95/1,5/1,5",60/50/55,Valid,4.15,0.074900,2.36,43.0,-,NaN,37.0,0.202,95.1,4.11,0.24,-,-,NaN,NaN,NaN
3,NaN,0.4,1.0,DI,-,95/15/80,"0,25/0,25/0,25","1,95/1,5/1,5",60/50/55,Valid,4.13,0.079358,2.98,28.0,13.81,0.083128,41.0,0.285,93.1,4.1,0.29,-,-,NaN,NaN,NaN
4,NaN,0.5,1.0,DI,-,95/15/80,"0,25/0,25/0,25","1,9/1,9/1,9",60/50/55,Valid,4.11,0.083816,3.08,25.0,-,NaN,37.0,0.285,93.1,4.08,0.3,-,-,NaN,NaN,NaN


# Clean

Wie man sehen kann ist das eingelesene Dataframe durch das Exportieren aus der für Menschen gut lesbaren Excel Datei ziemlich unstrukturiert. Daher muss man einige Schritte vornehmen um das Data Frame in eine verarbeitbare Form zu bringen:

1. Deutsche Bezeichner in englische Bezeichner umbennen und eindeutig Bezeichner einsetzen
2. Undefinierte Spalten löschen
3. Korrekte Datentypen für jede Spalte wählen


2. Jede Messung einer Messreihe zuordnen
3. Validity Spalte als erste Daten-Spalten definieren, damit man schnell sieht, ob ein Datenpunkt eine valide Messung darstellt
4. Alle Spalten ab der Spalte Test Remarks entferen, da diese keine relevanten Daten mehr enthalten


In [ ]:
# Local helper functions
def move_column(df, col_name, new_position):
    cols = list(df.columns)
    cols.insert(new_position, cols.pop(cols.index(col_name)))
    return df[cols]

# 1. Rename columns
df.rename(
    columns={
        "Unnamed: 0": "Series",
        "Test #": "Measurement",
        "Mol.\r\n(g/L)": "Molarity (g/l)",
        "H2O\r\n": "Water Type",
        "pH\r\n": "pH-Level (pH)",
        "Time\r\n(s)": "Time (s)",
        "Flow\r\n (L/min)": "Flow (l/min)",
        "Volt. Lim.\r\n(V)": "Voltage Limit (V)",
        "Curr. Lim.\r\n(A)": "Current Limit (A)",
        "Check": "Validity",
        "EC In \r\n(mS/cm)": "Undefined", #TODO: Add correct identifier
        "EC In Dev. (%)": "Undefined", #TODO: Add correct identifier
        "EC Pure\r\n(mS/cm)": "Undefined", #TODO: Add correct identifier
        "Remov. \r\n(%)": "Average CO2 Removal (%)",
        "SEC \r\n(kWh/m³)": "Undefined",
        "Spalte1": "Undefined", #TODO: Add correct identifier
        "Ry \r\n(%)": "Average Water Recovery (%)",
        "Min EC Pure (mS/cm)": "Undefined", #TODO: Add correct identifier
        "Max. Remov. (%)": "Maximum CO2 Removal (%)",
        "Avg. EC In (mS/cm)": "Undefined", #TODO: Add correct identifier
        "Avg. EC Pure (mS/cm)": "Undefined", #TODO: Add correct identifier
        "pH Pure": "Undefined", #TODO: Add correct identifier
        "pH Conc. ": "Undefined", #TODO: Add correct identifier
        "Test Remarks ": "Undefined",
        "Unnamed: 24": "Undefined",
        "Unnamed: 25": "Undefined",
    },
    inplace=True,
)

# 2. Delete columns that are undefined
df.drop(columns=["Undefined"], inplace=True)

# 2. Fill Series column
# df["Series"] = df["Series"].ffill()

# 3. Move Check column to be the third column
# df = move_column(df, "Check", 2)

df.head()

,Series,Measurement,Molarity (g/l),Water Type,pH-Level (pH),Time (s),Flow (l/min),Voltage Limit (V),Current Limit (A),Validity,Average CO2 Removal (%),Average Water Recovery (%),Maximum CO2 Removal (%)
0,0 - General Tests,0.1,1.0,DI,-,95/15/80,"0,45/0,5/0,5","1,95/1,5/1,5",60/50/55,Valid,71.0,40.0,98.1
1,NaN,0.2,1.0,DI,-,95/15/80,"0,35/0,4/0,4","1,95/1,5/1,5",60/50/55,Valid,57.0,41.0,96.8
2,NaN,0.3,1.0,DI,-,95/15/80,"0,3/0,3/0,3","1,95/1,5/1,5",60/50/55,Valid,43.0,37.0,95.1
3,NaN,0.4,1.0,DI,-,95/15/80,"0,25/0,25/0,25","1,95/1,5/1,5",60/50/55,Valid,28.0,41.0,93.1
4,NaN,0.5,1.0,DI,-,95/15/80,"0,25/0,25/0,25","1,9/1,9/1,9",60/50/55,Valid,25.0,37.0,93.1


In [ ]:
# Prepare input data

# df_input = df[["Series", "Measurement", "Flow\n (L/min)", "Curr. Lim.\n(A)"]].copy()

df_input[["Flow_C", "Flow_PP", "Flow_P"]] = df_input["Flow\n (L/min)"].str.split("/", expand=True)
df_input.drop(columns="Flow\n (L/min)", inplace=True)

df_input["Current_P"] = df_input["Curr. Lim.\n(A)"].str.split("/", expand=True)[2]
df_input.drop(columns="Curr. Lim.\n(A)", inplace=True)

columns_to_convert_input = ["Flow_C", "Flow_PP", "Flow_P", "Current_P"]
for col in columns_to_convert_input:
    df_input[col] = df_input[col].str.replace(',', '.', regex=False).astype(float)

In [ ]:
# Prepate output data

# df_output = df[["Remov. \n(%)", "Ry \n(%)"]].copy()
# df_output.rename(columns={"Remov. \n(%)": "Removal", "Ry \n(%)": "Recovery"}, inplace=True)

columns_to_convert_output = ["Removal", "Recovery"]
for col in columns_to_convert_output:
    df_output[col] = df_output[col].astype(float)

# Incorporate Expertise

An dieser Stelle wird gezielt Fachwissen des leitenden Doktoranten der Anlage und seines Teams eingesetzt, um bestimmte Datenpunkte gezielt auszuschließen, damit die Ergebnisse nicht durch fehlerhafte Werte verfälscht werden.

Folgende Messungen werden ausgeschlossen:
1. Alle als invalide gekennzeichnete Messwerte
2. Messungen deren Molarität in g/l unter 0.5 liegt
3. Solche Messungen bei denen kein destiliertes Wasser verwendet wurde
4. Die Gesamte 4. Messreihe, da in dieser Reihe Fehler beim Einstellen des ph Wertes passiert sind
5. Alle Messungen, deren Phasen-Zeiten von 155/45/120 abweichen, da sonst die Vergleichbarkeit der Messungen verloren geht

In [ ]:
# Drop all measurements that are not valid
df = df[df["Check"] == "Valid"]

# Drop all measurements with Mol.\n(g/L) unequal to 0.5
df = df[df["Mol.\n(g/L)"] == 0.5]

# Drop all measurements with H2O unequal to DI
df = df[df["H2O\n"] == "DI"]

# Drop the 4th measurement series
df = df[df["Series"] != "4 -  pH Series"]

# Drop all measurements with a time in seconds unequal to 155/45/120
df = df[df["Time\n(s)"] == "155/45/120"]

# Transform

one hot encoden

In [ ]:
def normalize(df, bounds):
    df_norm = df.copy()
    for col in bounds:
        a, b = bounds[col]
        df_norm[col] = (df[col] - a) / (b - a)
    return df_norm

input_bounds = {
    'Flow_C': (0.2, 1.5),
    'Flow_PP': (0.2, 1.5),
    'Flow_P': (0.2, 1.5),
    'Current_P': (35, 55)
}

output_bounds = {
    'Removal': (0, 100),
    'Recovery': (0, 100),
}

df_input_norm = normalize(df_input, input_bounds)
df_output_norm = normalize(df_output, output_bounds)


# als Bounds hier die maximalen und minimalen tatsächlich vorkommenden Werte der Messungen nehmen


# Restructure

In [ ]:
# Split into input und output DataFrame

# Validate

In [ ]:
# print plots usw

# Save

In [ ]:
processed_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed")
input_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "InputData.csv")
output_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "OutputData.csv")

df_input_norm.to_csv(input_data_path, index=False)
df_output_norm.to_csv(output_data_path, index=False)

In [ ]:
df_read_back_input = pd.read_csv(input_data_path)
df_read_back_input

In [ ]:
df_read_back_output = pd.read_csv(output_data_path)
df_read_back_output